In [ ]:
import io, os, time, json, subprocess
import numpy as np
from http.server import BaseHTTPRequestHandler, HTTPServer
from PIL import Image

SERVER_PORT = 8080
MODEL_DIR = '/home/xilinx/jupyter_notebooks/mobilenet'
PREPARE_PY = os.path.join(MODEL_DIR, 'prepare_image.py')
IMAGE_DAT = os.path.join(MODEL_DIR, 'image_int.dat')
RESULT_JSON = os.path.join(MODEL_DIR, 'result.json')
TMP_JPG = '/tmp/incoming.jpg'

def run_fpga_inference(jpeg_bytes):
    # 1. JPEG 저장
    with open(TMP_JPG, 'wb') as f:
        f.write(jpeg_bytes)
    
    # 2. prepare_image.py → image_int.dat
    subprocess.run(['python3', PREPARE_PY, TMP_JPG], 
                   cwd=MODEL_DIR, check=True)
    
    # 3. inference_server.py trigger (이미 떠있어야 함)
    if os.path.exists('/tmp/infer_done'):
        os.remove('/tmp/infer_done')
    open('/tmp/infer_request', 'w').close()
    
    # 4. 결과 대기
    timeout = 60
    start = time.time()
    while not os.path.exists('/tmp/infer_done'):
        if time.time() - start > timeout:
            raise TimeoutError("FPGA inference timeout")
        time.sleep(0.2)
    os.remove('/tmp/infer_done')
    
    # 5. 결과 읽기
    with open(RESULT_JSON, 'r', encoding='utf-8') as f:
        return json.load(f)


class ImageHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        if self.path != '/upload':
            self.send_response(404); self.end_headers()
            self.wfile.write(b'Not Found'); return
        
        content_length = int(self.headers.get('Content-Length', 0))
        if content_length == 0:
            self.send_response(400); self.end_headers()
            self.wfile.write(b'Empty body'); return
        
        # 본문 수신 (큰 이미지 안전하게)
        body = b''
        remaining = content_length
        while remaining > 0:
            chunk = self.rfile.read(min(4096, remaining))
            if not chunk: break
            body += chunk
            remaining -= len(chunk)
        
        if len(body) != content_length:
            self.send_response(400); self.end_headers()
            self.wfile.write(b'Incomplete'); return
        
        # FPGA 추론
        try:
            t0 = time.time()
            result = run_fpga_inference(body)
            elapsed = (time.time() - t0) * 1000
            
            top = result.get('top_class', '?')
            conf = result.get('confidence', 0)
            print(f"[INFER] {len(body):,}B | {elapsed:.0f}ms | {top} ({conf:.3f})")
            
            # ESP32에 결과 응답 (간단 포맷)
            response = f"{top},{conf:.4f}"
            
            # 또는 JSON 전체 반환하고 싶으면:
            # response = json.dumps(result, ensure_ascii=False)
            
            self.send_response(200)
            self.send_header('Content-Type', 'text/plain; charset=utf-8')
            self.end_headers()
            self.wfile.write(response.encode('utf-8'))
            
            # 로깅 (옵션)
            log_entry = {
                'timestamp': time.time(),
                'size_bytes': len(body),
                'elapsed_ms': elapsed,
                **result
            }
            with open('/home/xilinx/results.jsonl', 'a', encoding='utf-8') as f:
                f.write(json.dumps(log_entry, ensure_ascii=False) + '\n')
        
        except Exception as e:
            print(f"[ERROR] {e}")
            self.send_response(500); self.end_headers()
            self.wfile.write(f"ERROR: {e}".encode())
    
    def log_message(self, *a): pass  # 액세스 로그 끔


if __name__ == "__main__":
    print(f"[SERVER] FPGA inference server on port {SERVER_PORT}")
    print(f"  Make sure inference_server.py is also running (background)")
    server = HTTPServer(('0.0.0.0', SERVER_PORT), ImageHandler)
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        server.server_close()